In [ ]:
from itables import init_notebook_mode, show
import numpy as np
import torch
import torch.optim as optim
from torchviz import make_dot
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

import importlib
import aacbr_torch

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(aacbr_torch)

## Data Set

In [ ]:
# from ucimlrepo import fetch_ucirepo 
  
# # fetch dataset 
# connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# # data (as pandas dataframes) 
# X = connectionist_bench_sonar_mines_vs_rocks.data.features 
# y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

data = pd.read_csv('data/connectionist-bench-sonar-mines-vs-rocks/sonar.all-data')

data = data.values

X = np.array(data[:, :-1], dtype=np.float32)
y = np.array(data[:, -1])

show(X)
print(np.unique(y))



In [ ]:
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)


In [ ]:
print(encoder.classes_)
print(y)
print(type(y))

## Train Model

### Split into Training and Test Sets

In [ ]:
SEED = 42

In [ ]:
X, y = torch.tensor(X), torch.tensor(y, dtype=torch.float32)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=SEED)
print(f"Test Size:  {len(X_test)}")
print(f"Train Size:  {len(X_train)}")
print(f"Validation Size:  {len(X_val)}")



In [ ]:
print(X_train)

### Build AF

In [ ]:
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

In [ ]:
COMPARISON_FUNC = aacbr_torch.LearnedPartialOrder(X_train.shape[1])
PREPROCESS_FUNC = lambda x: x 

In [ ]:
DEFAULT_OUTCOME = 1
DEFAULT_CASE = means.clone()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
USE_SYMM_ATTACKS = False

In [ ]:
reload_imports()
def run_model(X_train, y_train, X_test, y_test, comparison_func = COMPARISON_FUNC, print_graph=False, print_matrix=False, print_compute=False, print_params=False, backward=False):
    X_train = PREPROCESS_FUNC(X_train)
    X_test = PREPROCESS_FUNC(X_test)
    default_case = PREPROCESS_FUNC(DEFAULT_CASE)
    
    model = aacbr_torch.AACBRTorch(X_train, y_train, comparison_func, default_case, 
                                   torch.tensor([DEFAULT_OUTCOME]), use_symmetric_attacks=USE_SYMM_ATTACKS)
    predicted = model(X_test)

    if print_graph:
        model.show_graph_with_labels()

    if print_matrix:
        model.show_matrix()
    
    criterion = torch.nn.BCELoss()
    loss = criterion(predicted.squeeze(), y_test)

    if print_compute:
        make_dot(loss, params=dict(model.named_parameters())).render("aacbr_torch", format="pdf")

    # if backward:
    #     loss.backward()
    
        

    if print_params:
        print("Predicted requires_grad", predicted.requires_grad)
        for name, param in model.named_parameters():
            print(f"Parameter: {name}, Shape: {param.shape}")
            print(param)

    y_test = y_test.detach().cpu().numpy()
    predicted = torch.round(predicted).detach().cpu().numpy()
    return([
        accuracy_score(y_test, predicted),
        precision_score(y_test, predicted),
        recall_score(y_test, predicted),
        f1_score(y_test, predicted)
    ])

### Cross-Validation

In [ ]:
# reload_imports()
# metrics = []
# for fold, (train_index,  val_index) in enumerate(kf.split(X_train)):
#     training_instances = X_train[train_index]
#     training_labels = y_train[train_index]
#     validation_instances = X_train[val_index]
#     validation_labels = y_train[val_index]


#     metrics.append(
#         run_model(training_instances, training_labels, validation_instances, validation_labels)
#     )

# print("Accuracy, Precision, Recall, F1")
# print(np.mean(metrics, axis=0))
# # for metric in metrics:
# #     print(metric)


### Validation Set

In [ ]:
import numpy as np
np.set_printoptions(threshold=10_000)
torch.set_printoptions(edgeitems=1000000)

In [ ]:
# run_model(X_train_full, y_train_full, X_val, y_val, print_matrix=True, print_compute=True)

## Train Loop

In [35]:
reload_imports()
torch.manual_seed(0) # TRY DIFFERENT INITIAL WEIGHTS 

# EPOCHS = 75 
EPOCHS = 5
default_case = PREPROCESS_FUNC(DEFAULT_CASE)
learned_order = aacbr_torch.LearnedPartialOrder(X_train.shape[1])

criterion = torch.nn.BCELoss()
                                   
model = aacbr_torch.AACBRTorch(X_train, y_train, learned_order, default_case, 
                            torch.tensor([DEFAULT_OUTCOME]), use_symmetric_attacks=USE_SYMM_ATTACKS)

optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

model.train()


AACBRTorch(
  (comparison_func): LearnedPartialOrder()
)

In [ ]:
learned_order.plot_parameters()

In [ ]:
# run_model(X_train_full, y_train_full, X_val, y_val, comparison_func=learned_order, print_matrix=True, print_compute=False)

In [27]:
torch.autograd.set_detect_anomaly(True)
# torch.autograd.set_detect_anomaly(False)

In [36]:
reload_imports()
def train():
    losses = []
    for fold, (casebase_index,  new_cases_index) in enumerate(kf.split(X_train)):

        print("Fold:", fold)

        # casebase = X_train[casebase_index]
        # casebase_labels = y_train[casebase_index]

        new_cases = X_train[new_cases_index]
        new_cases_labels = y_train[new_cases_index]

        model.set_casebase_indices(torch.tensor(casebase_index))
    


        # new_cases_labels = torch.tensor(new_cases_labels, dtype=torch.float32)

        pbar = tqdm(range(EPOCHS))

        for epoch in pbar:  

            running_loss = 0.0
            optimizer.zero_grad()

            predictions = model(new_cases).squeeze()


            loss = criterion(predictions, new_cases_labels)
            print("Loss requires_grad", loss.requires_grad)
            # loss.backward(retain_graph=True)
            print("LOSS:", loss.item())
            # make_dot(loss, params=dict(model.named_parameters())).render(f"aacbr_torch_fold{fold}_epoch{epoch}", format="pdf")
            loss.backward()

            print("Gradient: ", learned_order.W.grad)

            optimizer.step()

            running_loss += loss.item()
            losses.append(running_loss/len(new_cases))

            pbar.set_description(f'Epoch {epoch + 1}, Loss: {round(running_loss/len(new_cases), 4)}')




   

    print('Finished Training')

    plt.plot(losses)
    plt.show()

train()

Fold: 0


  0%|          | 0/5 [00:00<?, ?it/s]

tensor([[6.5684e-01, 1.0000e+00, 1.0000e+00, 0.0000e+00, 5.0000e-01, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 3.2799e-02, 1.0000e+00, 1.0000e+00,
         1.1921e-07, 0.0000e+00, 0.0000e+00, 2.3842e-07, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.1921e-07,
         1.0341e-03, 0.0000e+00, 1.0000e+00, 0.0000e+00, 9.9932e-01, 9.9909e-01,
         1.0000e+00, 2.3842e-07, 0.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 9.7715e-01, 1.0000e+00, 1.0000e+00, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 5.7543e-01,
         1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00,
         1.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00,
         1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00,
         1.0000e+00, 0.0000e

RuntimeError: where expected condition to be a boolean tensor, but got a tensor with dtype Float

In [ ]:
learned_order.plot_parameters()

In [ ]:

run_model(X_train_full, y_train_full, X_val, y_val, comparison_func=learned_order, print_matrix=True, print_compute=True)

### Test Set

In [ ]:
# reload_imports()
# print("Accuracy, Precision, Recall, F1")
# run_model(X_train_full, y_train_full, X_test, y_test, show_graph=False)